In [ ]:
import re
import nltk
import pandas as pd
from nltk.corpus import wordnet, stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import TweetTokenizer
from nltk import pos_tag
from nltk.tokenize import word_tokenize

# دانلود منابع NLTK (فقط یکبار)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('punkt_tab')
# خواندن داده‌ها
data = pd.read_csv('twits.csv')
final_results = pd.DataFrame({'tweet_id': data['tweet_id']})

# تابع پیش‌پردازش (حذف هوشمند اعداد)


def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    # بررسی وجود لینک
    has_url = bool(re.search(r'http\S+|www\S+|https\S+', text))
    # حذف لینک‌ها
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # حذف اعداد فقط اگر لینک وجود داشته باشد
    if has_url:
        text = re.sub(r'\b\d+\b', '', text)
    # حذف آیدی‌های عددی (@12345)
    text = re.sub(r'@\d+', '', text)
    # حذف علائم نگارشی
    text = re.sub(r'[^\w\s]', '', text)
    # حذف کاراکترهای غیرلاتین (شامل ایموجی و کاراکترهای خاص)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    # حذف فضاهای اضافی
    text = ' '.join(text.split())

    return text

data['preprocessed_text'] = data['text'].apply(preprocess_text)
final_results['preprocessed_text'] = data['preprocessed_text']


# توکن‌سازی با word_tokenize
def tokenize_text(text):
    if not text:
        return []
    return word_tokenize(text)

data['tokens'] = data['preprocessed_text'].apply(tokenize_text)
final_results['tokens'] = data['tokens'].apply(lambda x: ' '.join(x) if x else '')
# حذف کلمات توقف (با تطابق کوچک شده)
stop_words = set([word.lower() for word in stopwords.words('english')])
def remove_stopwords(tokens):
    if not tokens:
        return []
    return [word for word in tokens if word.lower() not in stop_words]

data['filtered_tokens'] = data['tokens'].apply(remove_stopwords)
final_results['filtered_tokens'] = data['filtered_tokens'].apply(lambda x: ' '.join(x) if x else '')

# تبدیل برچسب POS به فرمت wordnet
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

# برچسب‌گذاری POS
def pos_tagging(tokens):
    if not tokens:
        return []
    return pos_tag(tokens)

data['pos_tags'] = data['filtered_tokens'].apply(pos_tagging)
final_results['pos_tags'] = data['pos_tags'].apply(
    lambda x: ' '.join([f"{word}/{tag}" for word, tag in x]) if x else '')

# ریشه‌یابی (Stemming)
def stem_words(tokens):
    if not tokens:
        return []
    stemmer = PorterStemmer()
    return [stemmer.stem(word) for word in tokens]

data['stemmed_tokens'] = data['filtered_tokens'].apply(stem_words)
final_results['stemmed_tokens'] = data['stemmed_tokens'].apply(lambda x: ' '.join(x) if x else '')

# لماتیزاسیون با استفاده از برچسب POS دقیق
def lemmatize_words(tokens):
    if not tokens:
        return []
    lemmatizer = WordNetLemmatizer()
    pos_tags = pos_tag(tokens)
    return [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags]

data['lemmatized_tokens'] = data['filtered_tokens'].apply(lemmatize_words)
final_results['lemmatized_tokens'] = data['lemmatized_tokens'].apply(lambda x: ' '.join(x) if x else '')


In [ ]:
import zipfile

final_results.to_csv('final_results.csv', index=False)
result_files = ['final_results.csv', 'Text_Preprocessing_and_Analysis.ipynb']

with zipfile.ZipFile('submission.zip', 'w') as zipf:
    for file in result_files:
        zipf.write(file)

print("Results zipped successfully! You can now upload the file 'submission.zip'.")